In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE

df = pd.read_csv("Final_Depression1_anonymised.csv")
print("Shape:", df.shape)
df.head()
from sklearn.preprocessing import LabelEncoder
df_encoded = df.copy()

label_encoders = {}
target_column = 'Condition'

for col in df_encoded.select_dtypes(include='object').columns:
    if col != target_column:
        le = LabelEncoder()
        df_encoded[col] = le.fit_transform(df_encoded[col])
        label_encoders[col] = le

# Also encode the target column
le_target = LabelEncoder()
df_encoded[target_column] = le_target.fit_transform(df_encoded[target_column])

# Assume df_encoded is already loaded and preprocessed (tokenized + label encoded)

X = df_encoded.drop(columns=['Condition'])
y = df_encoded['Condition']

# Label encode target if not already
le_target = LabelEncoder()
y = le_target.fit_transform(y)

# Split
X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# SMOTE on training data
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)


Shape: (11670, 39)


In [2]:
X_train_tensor = torch.tensor(X_train_sm.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_sm, dtype=torch.long)
X_val_tensor = torch.tensor(X_val.values, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)


In [3]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_classes=5):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, num_classes)
        )
    
    def forward(self, x):
        return self.model(x)

input_dim = X_train_tensor.shape[1]
model = MLP(input_dim=input_dim)


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Training loop
epochs = 10
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")


Epoch 1/10 - Loss: 0.8053
Epoch 2/10 - Loss: 0.5335
Epoch 3/10 - Loss: 0.4826
Epoch 4/10 - Loss: 0.4539
Epoch 5/10 - Loss: 0.4393
Epoch 6/10 - Loss: 0.4284
Epoch 7/10 - Loss: 0.4001
Epoch 8/10 - Loss: 0.3903
Epoch 9/10 - Loss: 0.3769
Epoch 10/10 - Loss: 0.3752


In [6]:
model.eval()
with torch.no_grad():
    y_val_pred_logits = model(X_val_tensor)
    y_val_pred = torch.argmax(y_val_pred_logits, dim=1)

# Validation evaluation
val_acc = accuracy_score(y_val_tensor, y_val_pred)
print(f"✅ Validation Accuracy: {val_acc:.4f}\n")
print("📊 Classification Report (Validation):")
print(classification_report(y_val_tensor, y_val_pred, target_names=[str(c) for c in le_target.classes_]))

# Training evaluation
model.eval()
with torch.no_grad():
    y_train_pred = model(X_train_tensor)
    y_train_pred_labels = torch.argmax(y_train_pred, dim=1)
    train_acc = accuracy_score(y_train_tensor, y_train_pred_labels)

print(f"\n✅ Training Accuracy: {train_acc:.4f}")
print("📊 Classification Report (Training):")
print(classification_report(y_train_tensor, y_train_pred_labels, target_names=[str(c) for c in le_target.classes_]))



✅ Validation Accuracy: 0.8248

📊 Classification Report (Validation):
              precision    recall  f1-score   support

           0       0.90      0.97      0.94       794
           1       0.57      0.70      0.63       220
           2       0.81      0.68      0.74       432
           3       0.98      0.82      0.89       511
           4       0.68      0.76      0.72       377

    accuracy                           0.82      2334
   macro avg       0.79      0.79      0.78      2334
weighted avg       0.84      0.82      0.83      2334


✅ Training Accuracy: 0.8621
📊 Classification Report (Training):
              precision    recall  f1-score   support

           0       0.90      0.98      0.94      3173
           1       0.82      0.85      0.83      3173
           2       0.84      0.76      0.80      3173
           3       0.94      0.88      0.91      3173
           4       0.82      0.84      0.83      3173

    accuracy                           0.86     158